# Waddah MDD — CTC Phoneme Transcription Pipeline (V3)

> **HubertForCTC** fine-tuned on Arabic child speech for Mispronunciation Detection & Diagnosis.
>
> Runtime: Google Colab Pro (A100 40GB) — Drive mounted read-only for data.  
> All outputs are printed/displayed inline. Nothing is saved to disk.

## Phase 0 — Setup & Environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q torch==2.3.1 torchaudio==2.3.1
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.33.0
!pip install -q omegaconf==2.3.0 pandas==2.2.2 numpy==1.26.4
!pip install -q soundfile==0.12.1 librosa==0.10.2 jiwer==3.0.4
!pip install -q pyarabic==0.6.15 scikit-learn==1.5.1
!pip install -q matplotlib==3.9.2 seaborn==0.13.2
!pip install -q python-Levenshtein==0.25.1

In [ ]:
import os, sys, re, json, random, warnings, time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torchaudio
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

warnings.filterwarnings("ignore")

# === Seeds ===
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# === GPU check ===
assert torch.cuda.is_available(), "No GPU detected!"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# === PATHS (read-only — single source of truth) ===
AUDIO_DIR = "/content/drive/MyDrive/Segmented-Audios/phase0_output/preprocessed"
LEXICON_PATH = "/content/drive/MyDrive/Segmented-Audios/arabic_phoneme_lexicon.csv"

# Verify
wav_files = [f for f in os.listdir(AUDIO_DIR) if f.endswith('.wav')]
print(f"WAV files found: {len(wav_files)}")
assert len(wav_files) > 1000, f"Expected ~1713 files, found {len(wav_files)}"
assert os.path.exists(LEXICON_PATH), "Lexicon CSV not found!"
print("✅ All paths verified.")

---
## Phase 1 — Lexicon, Parser, Phonemizer, Aligner

### 1A — Arabic Normalizer

In [ ]:
def normalize_arabic(text: str) -> str:
    """Strip diacritics (U+064B-U+0652, U+0670), unify hamza forms, alif maksura."""
    text = re.sub(r'[\u064B-\u0652\u0670]', '', text)
    text = text.replace('أ', 'ا').replace('إ', 'ا').replace('آ', 'ا')
    text = text.replace('ى', 'ي')
    return text.strip()

# Quick test
assert normalize_arabic('أَحْمَر') == 'احمر'
print("✅ normalize_arabic works.")

### 1B — Lexicon Loader

In [ ]:
def load_lexicon(csv_path: str) -> dict:
    """Load arabic_phoneme_lexicon.csv → {normalized_word: {phoneme_sequence, target_phoneme, target_phoneme_idx, phoneme_position}}."""
    df = pd.read_csv(csv_path, encoding='utf-8')
    lexicon = {}
    all_phonemes = set()

    for _, row in df.iterrows():
        word = normalize_arabic(str(row['word']).strip())
        raw_seq = str(row['phoneme_sequence']).strip()
        # Parse slash-bounded tokens: "/ʔ/ /a/ /s/ /a/ /d/" → ["ʔ", "a", "s", "a", "d"]
        phonemes = [p.strip().strip('/') for p in raw_seq.split() if p.strip().strip('/')]
        all_phonemes.update(phonemes)

        target_ph = str(row['target_phoneme']).strip().strip('/')
        # Find target phoneme index in the sequence
        target_idx = None
        for i, ph in enumerate(phonemes):
            if ph == target_ph:
                target_idx = i
                break
        if target_idx is None:
            target_idx = 0  # fallback

        lexicon[word] = {
            "phoneme_sequence": phonemes,
            "target_phoneme": target_ph,
            "target_phoneme_idx": target_idx,
            "phoneme_position": str(row.get('phoneme_position', '')).strip(),
        }

    return lexicon, all_phonemes

word_to_phonemes, all_lexicon_phonemes = load_lexicon(LEXICON_PATH)
print(f"Lexicon loaded: {len(word_to_phonemes)} words")
print(f"Unique phonemes in lexicon: {len(all_lexicon_phonemes)}")
print(f"Phoneme set: {sorted(all_lexicon_phonemes)}")

### 1C — Filename Parser (DO NOT MODIFY)

In [ ]:
ARABIC_PATTERN = re.compile(r'[\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF]+')
FILENAME_HEAD_RE = re.compile(r'^(?P<diagnosis>AB|N)_(?P<id>ID\d+)_')

def parse_filename(filename: str, word_to_phonemes: dict) -> dict | None:
    """Parse '{AB|N}_{IDxx}_{w1}[_{w2}](N)?.wav' into structured fields.
    Disambiguates target vs. actually-said via lexicon membership — order-agnostic."""
    head = FILENAME_HEAD_RE.match(filename)
    if head is None:
        return None
    diagnosis, sample_id = head["diagnosis"], head["id"]
    stem = filename.replace(".wav", "").replace(" ", "")
    words = [normalize_arabic(w) for w in ARABIC_PATTERN.findall(stem) if w.strip()]
    if not words:
        return None
    if len(words) == 1:
        w = words[0]
        if w not in word_to_phonemes:
            return None
        return {"diagnosis": diagnosis, "id": sample_id,
                "target": w, "actual": w, "kind": "target_only"}
    if len(words) == 2:
        w1, w2 = words
        if w1 == w2:
            if w1 not in word_to_phonemes:
                return None
            return {"diagnosis": diagnosis, "id": sample_id,
                    "target": w1, "actual": w1, "kind": "duplicated"}
        if w1 in word_to_phonemes and w2 not in word_to_phonemes:
            return {"diagnosis": diagnosis, "id": sample_id,
                    "target": w1, "actual": w2, "kind": "target_actual"}
        if w2 in word_to_phonemes and w1 not in word_to_phonemes:
            return {"diagnosis": diagnosis, "id": sample_id,
                    "target": w2, "actual": w1, "kind": "target_actual"}
    return None  # malformed

print("✅ parse_filename defined.")

### 1D — Phonemizer (2-tier: lexicon → letter-G2P)

In [ ]:
LETTER_TO_PHONEME = {
    'ب':'b',  'ت':'t',  'ث':'θ',  'ج':'dʒ', 'ح':'ħ',  'خ':'χ',
    'د':'d',  'ذ':'ð',  'ر':'r',  'ز':'z',  'س':'s',  'ش':'ʃ',
    'ص':'sˤ', 'ض':'dˤ', 'ط':'tˤ', 'ظ':'ðˤ', 'ع':'ʕ',  'غ':'γ',
    'ف':'f',  'ق':'q',  'ك':'k',  'ل':'l',  'م':'m',  'ن':'n',
    'ه':'h',  'و':'uː', 'ي':'iː', 'ا':'aː',
    'ى':'aː', 'ء':'ʔ', 'أ':'ʔ', 'إ':'ʔ', 'آ':'ʔ', 'ة':'a',
}

def get_phonemes(word: str, lexicon: dict) -> tuple:
    """Return (phoneme_list, source) where source ∈ {'lexicon', 'letter_g2p'}."""
    norm = normalize_arabic(word)
    if norm in lexicon:
        return lexicon[norm]["phoneme_sequence"], "lexicon"
    return [LETTER_TO_PHONEME[c] for c in norm if c in LETTER_TO_PHONEME], "letter_g2p"

print("✅ get_phonemes defined.")

### 1E — Levenshtein Aligner

In [ ]:
def levenshtein_align(ref: list, hyp: list):
    """DP-based Levenshtein alignment.
    Returns (subs, dels, ins, ops) where ops is list of (op_type, ref_ph, hyp_ph).
    op_type ∈ {'correct', 'substitution', 'deletion', 'insertion'}."""
    n, m = len(ref), len(hyp)
    # DP table
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref[i - 1] == hyp[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j - 1],  # sub
                                    dp[i - 1][j],      # del
                                    dp[i][j - 1])      # ins

    # Backtrace
    ops = []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref[i - 1] == hyp[j - 1]:
            ops.append(("correct", ref[i - 1], hyp[j - 1]))
            i -= 1
            j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:
            ops.append(("substitution", ref[i - 1], hyp[j - 1]))
            i -= 1
            j -= 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            ops.append(("deletion", ref[i - 1], None))
            i -= 1
        elif j > 0 and dp[i][j] == dp[i][j - 1] + 1:
            ops.append(("insertion", None, hyp[j - 1]))
            j -= 1
        else:
            break

    ops.reverse()

    subs = sum(1 for o in ops if o[0] == "substitution")
    dels = sum(1 for o in ops if o[0] == "deletion")
    ins_ = sum(1 for o in ops if o[0] == "insertion")

    return subs, dels, ins_, ops

print("✅ levenshtein_align defined.")

### 1F — Phase 1 Tests

In [ ]:
# --- Test 1: Filename parser on real examples ---
test_cases = [
    ("N_ID039_بيض_بيض(1).wav", "N", "بيض", "بيض"),
    ("N_ID039_تلفون_تلفون.wav", "N", "تلفون", "تلفون"),
    ("AB_ID001_احمر_اخمر.wav", "AB", "احمر", "اخمر"),
    ("AB_ID040_خبز_خبزه.wav", "AB", "خبز", "خبزه"),
    ("AB_ID040_كورة_كرة.wav", "AB", "كرة", "كورة"),
    ("N_ID040_اذن_اذن.wav", "N", "اذن", "اذن"),
]

print("=== Filename Parser Tests ===")
for fname, exp_diag, exp_target, exp_actual in test_cases:
    result = parse_filename(fname, word_to_phonemes)
    if result is None:
        print(f"  ⚠️  {fname} → None (expected non-None, target may not be in lexicon — check lexicon)")
        continue
    target_norm = normalize_arabic(exp_target)
    actual_norm = normalize_arabic(exp_actual)
    assert result["diagnosis"] == exp_diag, f"Diagnosis mismatch for {fname}"
    assert result["target"] == target_norm, f"Target mismatch for {fname}: got {result['target']}, expected {target_norm}"
    assert result["actual"] == actual_norm, f"Actual mismatch for {fname}: got {result['actual']}, expected {actual_norm}"
    print(f"  ✅ {fname} → diag={result['diagnosis']}, target={result['target']}, actual={result['actual']}, kind={result['kind']}")

# --- Test 2: Malformed cases → None ---
print("\n=== Malformed Cases ===")
malformed = ["random_file.wav", "X_ID001_بيض.wav", "something.mp3"]
for fname in malformed:
    result = parse_filename(fname, word_to_phonemes)
    assert result is None, f"Expected None for {fname}"
    print(f"  ✅ {fname} → None (correct)")

# --- Test 3: Phonemizer ---
print("\n=== Phonemizer Tests ===")
ph, src = get_phonemes("اخمر", word_to_phonemes)
assert len(ph) > 0 and src == "letter_g2p", f"Expected letter_g2p for non-word"
print(f"  ✅ 'اخمر' → {ph} (source: {src})")

# Lexicon word
for test_word in list(word_to_phonemes.keys())[:3]:
    ph, src = get_phonemes(test_word, word_to_phonemes)
    assert src == "lexicon"
    print(f"  ✅ '{test_word}' → {ph} (source: {src})")

# --- Test 4: Aligner ---
print("\n=== Aligner Tests ===")
s, d, i, ops = levenshtein_align(list("abc"), list("abc"))
assert s == 0 and d == 0 and i == 0, "Identical should be 0 cost"
print(f"  ✅ abc vs abc → subs={s}, dels={d}, ins={i}")

s, d, i, ops = levenshtein_align(list("abc"), list("axc"))
assert s == 1 and d == 0 and i == 0, "Single sub"
print(f"  ✅ abc vs axc → subs={s}, dels={d}, ins={i}")

s, d, i, ops = levenshtein_align(list("abc"), list("ac"))
assert d == 1, "Single deletion"
print(f"  ✅ abc vs ac → subs={s}, dels={d}, ins={i}")

s, d, i, ops = levenshtein_align(list("abc"), list("abxc"))
assert i == 1, "Single insertion"
print(f"  ✅ abc vs abxc → subs={s}, dels={d}, ins={i}")

# Cross-check with python-Levenshtein
import Levenshtein as lev
for ref_str, hyp_str in [("kitten", "sitting"), ("saturday", "sunday"), ("abc", "abc")]:
    ref_l, hyp_l = list(ref_str), list(hyp_str)
    s, d, i, _ = levenshtein_align(ref_l, hyp_l)
    expected = lev.distance(ref_str, hyp_str)
    assert s + d + i == expected, f"Cost mismatch: {s+d+i} vs {expected} for {ref_str}/{hyp_str}"
    print(f"  ✅ '{ref_str}' vs '{hyp_str}' → cost={s+d+i} (matches python-Levenshtein)")

print("\n✅ All Phase 1 tests passed.")

---
## Phase 2 — Data Manifest & Sanity Report

### 2A — Build Manifest

In [ ]:
def build_manifest(audio_dir: str, lexicon: dict) -> tuple:
    """Walk audio dir, parse filenames, build manifest DataFrame.
    Returns (manifest_df, malformed_list)."""
    records = []
    malformed = []

    all_files = sorted([f for f in os.listdir(audio_dir) if f.endswith('.wav')])
    print(f"Scanning {len(all_files)} WAV files...")

    for fname in all_files:
        parsed = parse_filename(fname, lexicon)
        if parsed is None:
            # Determine reason
            head = FILENAME_HEAD_RE.match(fname)
            if head is None:
                reason = "no AB/N_IDxxx prefix"
            else:
                stem = fname.replace(".wav", "").replace(" ", "")
                words = [normalize_arabic(w) for w in ARABIC_PATTERN.findall(stem) if w.strip()]
                if not words:
                    reason = "no Arabic words found"
                elif len(words) == 1 and words[0] not in lexicon:
                    reason = f"single word '{words[0]}' not in lexicon"
                elif len(words) == 2:
                    w1, w2 = words
                    in1 = w1 in lexicon
                    in2 = w2 in lexicon
                    if in1 and in2:
                        reason = f"both words in lexicon: '{w1}', '{w2}'"
                    elif not in1 and not in2:
                        reason = f"neither word in lexicon: '{w1}', '{w2}'"
                    else:
                        reason = f"unexpected 2-word case: '{w1}'(in={in1}), '{w2}'(in={in2})"
                else:
                    reason = f"{len(words)} Arabic words found"
            malformed.append({"filename": fname, "reason": reason})
            continue

        filepath = os.path.join(audio_dir, fname)

        # Get phonemes
        target_phonemes = lexicon[parsed["target"]]["phoneme_sequence"]
        actual_phonemes, actual_source = get_phonemes(parsed["actual"], lexicon)
        target_phoneme = lexicon[parsed["target"]]["target_phoneme"]
        target_phoneme_idx = lexicon[parsed["target"]]["target_phoneme_idx"]

        records.append({
            "filepath": filepath,
            "filename": fname,
            "id": parsed["id"],
            "diagnosis": parsed["diagnosis"],
            "kind": parsed["kind"],
            "target": parsed["target"],
            "actual": parsed["actual"],
            "target_phonemes": target_phonemes,
            "actual_phonemes": actual_phonemes,
            "actual_phoneme_source": actual_source,
            "target_phoneme": target_phoneme,
            "target_phoneme_idx": target_phoneme_idx,
        })

    df = pd.DataFrame(records)
    print(f"Parsed: {len(df)} samples | Malformed: {len(malformed)} files")
    return df, malformed

manifest_df, malformed_list = build_manifest(AUDIO_DIR, word_to_phonemes)

### 2B — Word-Stratified Split (80/10/10)

In [ ]:
def stratified_split(df, train_ratio=0.8, val_ratio=0.1, seed=42):
    """Group samples by target word, shuffle each group, split per-word."""
    rng = random.Random(seed)
    train_idx, val_idx, test_idx = [], [], []
    for word, group in df.groupby("target"):
        indices = group.index.tolist()
        rng.shuffle(indices)
        n = len(indices)
        n_train = max(1, int(n * train_ratio))
        n_val = max(0, int(n * val_ratio))
        train_idx.extend(indices[:n_train])
        val_idx.extend(indices[n_train:n_train + n_val])
        test_idx.extend(indices[n_train + n_val:])
    df.loc[train_idx, "split"] = "train"
    df.loc[val_idx, "split"] = "val"
    df.loc[test_idx, "split"] = "test"
    return df

manifest_df = stratified_split(manifest_df, seed=SEED)
print("✅ Splits assigned.")

### 2C — Sanity Report

In [ ]:
print("=" * 60)
print("           DATA SANITY REPORT")
print("=" * 60)

print(f"\nTotal WAV files scanned:  {len(wav_files)}")
print(f"Successfully parsed:      {len(manifest_df)}")
print(f"Malformed (skipped):      {len(malformed_list)}")
print(f"Match rate:               {len(manifest_df)/len(wav_files)*100:.1f}%")

print(f"\n--- Class Distribution (Total) ---")
diag_counts = manifest_df['diagnosis'].value_counts()
for label, count in diag_counts.items():
    print(f"  {label}: {count} ({count/len(manifest_df)*100:.1f}%)")

print(f"\n--- Split Sizes ---")
for split in ['train', 'val', 'test']:
    split_df = manifest_df[manifest_df['split'] == split]
    n_count = len(split_df[split_df['diagnosis'] == 'N'])
    ab_count = len(split_df[split_df['diagnosis'] == 'AB'])
    print(f"  {split:5s}: {len(split_df):5d} total | N={n_count}, AB={ab_count}")

print(f"\n--- Phonemizer Source Breakdown (AB samples) ---")
ab_df = manifest_df[manifest_df['diagnosis'] == 'AB']
source_counts = ab_df['actual_phoneme_source'].value_counts()
for src, count in source_counts.items():
    print(f"  {src}: {count} ({count/len(ab_df)*100:.1f}%)")

print(f"\n--- Kind Distribution ---")
kind_counts = manifest_df['kind'].value_counts()
for kind, count in kind_counts.items():
    print(f"  {kind}: {count}")

# Check for known anomaly
kasra_files = [m for m in malformed_list if 'ID009' in m['filename'] and 'حصان' in m['filename']]
if kasra_files:
    print(f"\n⚠️  Known anomaly found: {kasra_files[0]['filename']} — {kasra_files[0]['reason']}")
else:
    print(f"\nℹ️  Known kasra anomaly (ِAB_ID009_حصان_حشان.wav) not found in malformed list.")

if malformed_list:
    print(f"\n--- Malformed Files ({len(malformed_list)}) ---")
    for m in malformed_list:
        print(f"  {m['filename']}")
        print(f"    Reason: {m['reason']}")

print("\n" + "=" * 60)

# Show sample rows
print("\n--- Sample Manifest Rows ---")
display(manifest_df.head(10))

---
## Phase 3 — Vocab, Processor, Dataset, Collator

### 3A — Build Vocab

In [ ]:
# Collect all phonemes from lexicon + LETTER_TO_PHONEME
all_phonemes = set(all_lexicon_phonemes)
all_phonemes.update(LETTER_TO_PHONEME.values())

# Build vocab dict
vocab_dict = {"<pad>": 0, "<unk>": 1, "|": 2}
for i, ph in enumerate(sorted(all_phonemes), start=3):
    vocab_dict[ph] = i

print(f"Vocab size: {len(vocab_dict)}")
if len(vocab_dict) != 38:
    print(f"⚠️  Expected 38 tokens, got {len(vocab_dict)}")
print(f"Vocab: {vocab_dict}")

### 3B — Processor

In [ ]:
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

# Tokenizer requires a file — use /tmp/ temporarily
vocab_path = "/tmp/waddah_vocab.json"
with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False)

tokenizer = Wav2Vec2CTCTokenizer(
    vocab_path,
    unk_token="<unk>",
    pad_token="<pad>",
    word_delimiter_token="|",
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

os.remove(vocab_path)
print(f"✅ Processor built. Tokenizer vocab size: {tokenizer.vocab_size}")

### 3C — Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader

MAX_AUDIO_LENGTH = 5  # seconds
MAX_SAMPLES = MAX_AUDIO_LENGTH * 16000  # 80,000 samples

class WaddahCTCDataset(Dataset):
    """CTC dataset: loads audio, returns processed waveform + phoneme token IDs."""

    def __init__(self, df, processor, tokenizer, max_samples=MAX_SAMPLES):
        self.samples = df.reset_index(drop=True)
        self.processor = processor
        self.tokenizer = tokenizer
        self.max_samples = max_samples
        self._bad_files = []

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        row = self.samples.iloc[idx]

        # Load audio
        try:
            waveform, sr = torchaudio.load(row["filepath"])
        except Exception as e:
            print(f"⚠️ Error loading {row['filename']}: {e}")
            # Return a tiny silent waveform as fallback
            waveform = torch.zeros(1, 16000)
            sr = 16000

        # Resample if needed
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(sr, 16000)
            waveform = resampler(waveform)

        # Mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Truncate to max length
        if waveform.shape[1] > self.max_samples:
            waveform = waveform[:, :self.max_samples]

        # Process with feature extractor
        input_values = self.processor.feature_extractor(
            waveform.squeeze().numpy(),
            sampling_rate=16000,
            return_tensors="np",
        ).input_values[0]

        # Encode labels: actual_phonemes → token IDs
        actual_phonemes = row["actual_phonemes"]
        # Join phonemes with space for tokenizer (it splits on space by default)
        label_str = " ".join(actual_phonemes)
        labels = self.tokenizer(label_str, return_tensors="np").input_ids[0]

        return {
            "input_values": input_values,
            "labels": labels.tolist(),
        }

print("✅ WaddahCTCDataset defined.")

### 3D — Data Collator

In [ ]:
from dataclasses import dataclass
from typing import Any

@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: bool = True

    def __call__(self, features):
        input_values = [{"input_values": f["input_values"]} for f in features]
        labels = [f["labels"] for f in features]

        batch = self.processor.feature_extractor.pad(
            input_values,
            padding=True,
            return_tensors="pt",
        )

        labels_padded = torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(l, dtype=torch.long) for l in labels],
            batch_first=True,
            padding_value=-100,
        )
        batch["labels"] = labels_padded
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)
print("✅ DataCollatorCTCWithPadding defined.")

### 3E — Create Datasets & Sanity Check

In [ ]:
# Create datasets
train_df = manifest_df[manifest_df['split'] == 'train']
val_df = manifest_df[manifest_df['split'] == 'val']
test_df = manifest_df[manifest_df['split'] == 'test']

train_dataset = WaddahCTCDataset(train_df, processor, tokenizer)
val_dataset = WaddahCTCDataset(val_df, processor, tokenizer)
test_dataset = WaddahCTCDataset(test_df, processor, tokenizer)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# Sanity: load one batch
loader = DataLoader(train_dataset, batch_size=2, collate_fn=data_collator)
batch = next(iter(loader))
print(f"\nBatch shapes:")
print(f"  input_values: {batch['input_values'].shape}")
print(f"  labels:       {batch['labels'].shape}")

# Round-trip check: phonemes → IDs → phonemes
sample = train_dataset[0]
label_ids = sample["labels"]
decoded = tokenizer.decode(label_ids, skip_special_tokens=True)
original = " ".join(train_df.iloc[0]["actual_phonemes"])
print(f"\nRound-trip check:")
print(f"  Original phonemes: {original}")
print(f"  Token IDs:         {label_ids}")
print(f"  Decoded back:      {decoded}")
print(f"  Match: {'✅' if original == decoded else '⚠️ mismatch — check tokenizer'}")

---
## Phase 4 — Model Setup

In [ ]:
from transformers import HubertForCTC
import transformers

transformers.set_seed(SEED)

model = HubertForCTC.from_pretrained(
    "omarxadel/hubert-large-arabic-egyptian",
    vocab_size=len(vocab_dict),
    ignore_mismatched_sizes=True,
    pad_token_id=tokenizer.pad_token_id,
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.1,
    ctc_loss_reduction="mean",
)

# Freeze ONLY the CNN feature extractor
model.freeze_feature_encoder()

# Parameter summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Total parameters:     {total_params:>12,}")
print(f"Trainable parameters: {trainable_params:>12,}")
print(f"Frozen parameters:    {frozen_params:>12,}")

In [ ]:
# Forward pass sanity check
model_gpu = model.to("cuda")
dummy = torch.randn(2, 16000).to("cuda")
with torch.no_grad():
    out = model_gpu(dummy)
print(f"Logits shape: {out.logits.shape}")
assert not torch.isnan(out.logits).any(), "NaN in logits!"
print("✅ Forward pass OK — no NaN.")
model = model_gpu.cpu()
torch.cuda.empty_cache()

---
## Phase 5 — Training

### 5A — Metrics Function

In [ ]:
def compute_per(pred):
    """Compute Phoneme Error Rate during validation."""
    pred_ids = pred.predictions.argmax(-1)
    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_strs = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_strs = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    total_ref, total_errors = 0, 0
    for pred_str, label_str in zip(pred_strs, label_strs):
        pred_ph = pred_str.split()
        label_ph = label_str.split()
        if not label_ph:
            continue
        s, d, i, _ = levenshtein_align(label_ph, pred_ph)
        total_errors += s + d + i
        total_ref += len(label_ph)

    per = total_errors / total_ref if total_ref > 0 else 1.0
    return {"per": per}

print("✅ compute_per defined.")

### 5B — Smoke Test (16 samples × 1 epoch)

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from torch.utils.data import Subset

# Tiny subsets for smoke test
smoke_train = Subset(train_dataset, list(range(min(16, len(train_dataset)))))
smoke_val = Subset(val_dataset, list(range(min(8, len(val_dataset)))))

smoke_args = TrainingArguments(
    output_dir="/tmp/smoke-test",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    eval_strategy="epoch",
    fp16=True,
    learning_rate=3e-4,
    logging_steps=1,
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
)

# Fresh model for smoke test
smoke_model = HubertForCTC.from_pretrained(
    "omarxadel/hubert-large-arabic-egyptian",
    vocab_size=len(vocab_dict),
    ignore_mismatched_sizes=True,
    pad_token_id=tokenizer.pad_token_id,
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.1,
    ctc_loss_reduction="mean",
)
smoke_model.freeze_feature_encoder()

smoke_trainer = Trainer(
    model=smoke_model,
    args=smoke_args,
    train_dataset=smoke_train,
    eval_dataset=smoke_val,
    data_collator=data_collator,
    compute_metrics=compute_per,
)

print("Running smoke test...")
smoke_result = smoke_trainer.train()
smoke_eval = smoke_trainer.evaluate()
print(f"\nSmoke test results:")
print(f"  Train loss: {smoke_result.training_loss:.4f}")
print(f"  Val PER:    {smoke_eval['eval_per']:.4f}")
print("✅ Smoke test passed — no errors.")

# Cleanup
del smoke_model, smoke_trainer
torch.cuda.empty_cache()

### 5C — Full Training (30 epochs)

In [ ]:
transformers.set_seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

training_args = TrainingArguments(
    output_dir="/tmp/waddah-v3-checkpoints",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,        # effective batch = 16
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=30,
    fp16=True,
    learning_rate=3e-4,
    warmup_ratio=0.2,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="per",
    greater_is_better=False,
    save_total_limit=2,
    dataloader_num_workers=2,
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_per,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

print("Starting full training...")
start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time

print(f"\n{'=' * 60}")
print(f"  Training complete!")
print(f"  Duration:       {elapsed/60:.1f} minutes")
print(f"  Final train loss: {train_result.training_loss:.4f}")

# Best val PER
val_result = trainer.evaluate()
print(f"  Best val PER:   {val_result['eval_per']:.4f}")

if val_result['eval_per'] > 0.30:
    print("\n⚠️  WARNING: PER > 0.30 — something may be wrong!")
print(f"{'=' * 60}")

---
## Phase 6 — Evaluation

### 6A — Decode Test Set

In [ ]:
def diagnose_at_target(predicted_phonemes, canonical_phonemes, target_idx):
    """Return the operation at the target phoneme position."""
    _, _, _, ops = levenshtein_align(canonical_phonemes, predicted_phonemes)
    canonical_pos = 0
    for op_type, ref_ph, hyp_ph in ops:
        if op_type in ("correct", "substitution", "deletion"):
            if canonical_pos == target_idx:
                return op_type
            canonical_pos += 1
    return "insertion"

def decode_predictions(trainer, dataset, df):
    """Run CTC decode on all samples, return detailed results."""
    predictions = trainer.predict(dataset)
    pred_ids = predictions.predictions.argmax(-1)

    results = []
    for i in range(len(df)):
        row = df.iloc[i]

        # Decode predicted phonemes
        pred_tokens = tokenizer.decode(pred_ids[i], skip_special_tokens=True)
        pred_phonemes = pred_tokens.split() if pred_tokens.strip() else []

        canonical = row["target_phonemes"]
        diagnosis = row["diagnosis"]
        target_ph_idx = row["target_phoneme_idx"]

        # PER for this sample
        s, d, ins, ops = levenshtein_align(canonical, pred_phonemes)
        per = (s + d + ins) / len(canonical) if len(canonical) > 0 else 0.0

        # Whole-word detection: AB if any op is not "correct"
        has_error = any(o[0] != "correct" for o in ops)
        pred_detection = "AB" if has_error else "N"

        # Target-only detection
        target_op = diagnose_at_target(pred_phonemes, canonical, target_ph_idx)
        pred_target_det = "AB" if target_op in ("substitution", "deletion") else "N"

        # Gold diagnosis for AB samples (from aligning actual vs canonical)
        gold_diagnosis_type = None
        if diagnosis == "AB":
            gold_actual = row["actual_phonemes"]
            _, _, _, gold_ops = levenshtein_align(canonical, gold_actual)
            gc = 0
            for op_t, _, _ in gold_ops:
                if op_t in ("correct", "substitution", "deletion"):
                    if gc == target_ph_idx:
                        gold_diagnosis_type = op_t
                        break
                    gc += 1
            if gold_diagnosis_type is None:
                gold_diagnosis_type = "insertion"

        results.append({
            "filename": row["filename"],
            "id": row["id"],
            "diagnosis_gold": diagnosis,
            "target_word": row["target"],
            "canonical": canonical,
            "predicted": pred_phonemes,
            "actual_phonemes": row["actual_phonemes"],
            "per": per,
            "subs": s, "dels": d, "ins": ins,
            "pred_detection": pred_detection,
            "pred_target_det": pred_target_det,
            "target_op": target_op,
            "gold_diagnosis_type": gold_diagnosis_type,
            "target_phoneme": row["target_phoneme"],
            "target_phoneme_idx": target_ph_idx,
        })

    return pd.DataFrame(results)

print("Decoding test set...")
test_results = decode_predictions(trainer, test_dataset, test_df.reset_index(drop=True))
print(f"✅ Decoded {len(test_results)} test samples.")

### 6B — Compute & Print All Metrics

In [ ]:
# === Global PER ===
total_errors = test_results['subs'].sum() + test_results['dels'].sum() + test_results['ins'].sum()
total_ref = test_results['canonical'].apply(len).sum()
global_per = total_errors / total_ref if total_ref > 0 else 0

# PER by class
n_mask = test_results['diagnosis_gold'] == 'N'
ab_mask = test_results['diagnosis_gold'] == 'AB'

n_errors = test_results.loc[n_mask, 'subs'].sum() + test_results.loc[n_mask, 'dels'].sum() + test_results.loc[n_mask, 'ins'].sum()
n_ref = test_results.loc[n_mask, 'canonical'].apply(len).sum()
per_normal = n_errors / n_ref if n_ref > 0 else 0

ab_errors = test_results.loc[ab_mask, 'subs'].sum() + test_results.loc[ab_mask, 'dels'].sum() + test_results.loc[ab_mask, 'ins'].sum()
ab_ref = test_results.loc[ab_mask, 'canonical'].apply(len).sum()
per_abnormal = ab_errors / ab_ref if ab_ref > 0 else 0

per_gap = per_abnormal - per_normal

# WER (whole-sequence error rate)
wer = (test_results['per'] > 0).mean()

# === Whole-word detection ===
tp = ((test_results['pred_detection'] == 'AB') & (test_results['diagnosis_gold'] == 'AB')).sum()
fp = ((test_results['pred_detection'] == 'AB') & (test_results['diagnosis_gold'] == 'N')).sum()
tn = ((test_results['pred_detection'] == 'N') & (test_results['diagnosis_gold'] == 'N')).sum()
fn = ((test_results['pred_detection'] == 'N') & (test_results['diagnosis_gold'] == 'AB')).sum()

det_acc = (tp + tn) / (tp + fp + tn + fn) if (tp + fp + tn + fn) > 0 else 0
det_prec = tp / (tp + fp) if (tp + fp) > 0 else 0
det_rec = tp / (tp + fn) if (tp + fn) > 0 else 0
det_f1 = 2 * det_prec * det_rec / (det_prec + det_rec) if (det_prec + det_rec) > 0 else 0

# === Target-only detection ===
tp_t = ((test_results['pred_target_det'] == 'AB') & (test_results['diagnosis_gold'] == 'AB')).sum()
fp_t = ((test_results['pred_target_det'] == 'AB') & (test_results['diagnosis_gold'] == 'N')).sum()
tn_t = ((test_results['pred_target_det'] == 'N') & (test_results['diagnosis_gold'] == 'N')).sum()
fn_t = ((test_results['pred_target_det'] == 'N') & (test_results['diagnosis_gold'] == 'AB')).sum()

tdet_prec = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0
tdet_rec = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0
tdet_f1 = 2 * tdet_prec * tdet_rec / (tdet_prec + tdet_rec) if (tdet_prec + tdet_rec) > 0 else 0

# === Diagnosis accuracy ===
ab_results = test_results[ab_mask]
if len(ab_results) > 0 and ab_results['gold_diagnosis_type'].notna().any():
    diag_correct = (ab_results['target_op'] == ab_results['gold_diagnosis_type']).sum()
    diag_acc = diag_correct / len(ab_results)
else:
    diag_acc = 0

# === PRINT HEADLINE REPORT ===
print("=" * 60)
print(f"Normal samples:    {n_mask.sum()}")
print(f"Abnormal samples:  {ab_mask.sum()}")
print(f"Normal avg PER:    {per_normal*100:.2f}%")
print(f"Abnormal avg PER:  {per_abnormal*100:.2f}%")
print(f"PER gap:           {per_gap*100:.2f}%")
print(f"Detection acc:     {det_acc*100:.2f}%")
print(f"Diagnosis acc:     {diag_acc*100:.2f}%")
print(f"WER:               {wer*100:.2f}%")
print(f"Precision:         {det_prec*100:.2f}%")
print(f"Recall:            {det_rec*100:.2f}%")
print(f"F1:                {det_f1*100:.2f}%")
print("=" * 60)

# Whole-word confusion matrix
print(f"\n--- Whole-Word Detection Confusion Matrix ---")
print(f"  TP (AB→AB): {tp} | FP (N→AB): {fp}")
print(f"  FN (AB→N):  {fn} | TN (N→N):  {tn}")

# Target-only detection
print(f"\n--- Target-Only Detection ---")
print(f"  Precision: {tdet_prec*100:.2f}%")
print(f"  Recall:    {tdet_rec*100:.2f}%")
print(f"  F1:        {tdet_f1*100:.2f}%")

# Diagnosis breakdown for AB samples
print(f"\n--- Diagnosis Breakdown (AB samples, predicted) ---")
if len(ab_results) > 0:
    for op_type in ['correct', 'substitution', 'deletion', 'insertion']:
        count = (ab_results['target_op'] == op_type).sum()
        print(f"  {op_type}: {count}")

### 6C — Per-Phoneme Error Table

In [ ]:
# Top-15 most mispronounced phonemes
phoneme_errors = defaultdict(lambda: {"total": 0, "subs": 0, "dels": 0, "ins": 0})

for _, row in test_results.iterrows():
    canonical = row["canonical"]
    predicted = row["predicted"]
    _, _, _, ops = levenshtein_align(canonical, predicted)
    for op_type, ref_ph, hyp_ph in ops:
        if op_type == "substitution" and ref_ph:
            phoneme_errors[ref_ph]["total"] += 1
            phoneme_errors[ref_ph]["subs"] += 1
        elif op_type == "deletion" and ref_ph:
            phoneme_errors[ref_ph]["total"] += 1
            phoneme_errors[ref_ph]["dels"] += 1
        elif op_type == "insertion" and hyp_ph:
            phoneme_errors[hyp_ph]["total"] += 1
            phoneme_errors[hyp_ph]["ins"] += 1

# Sort by total errors
sorted_ph = sorted(phoneme_errors.items(), key=lambda x: x[1]["total"], reverse=True)[:15]

print("\n--- Top 15 Mispronounced Phonemes ---")
print(f"{'Phoneme':<10} {'Errors':<8} {'Subs':<6} {'Dels':<6} {'Ins':<6}")
print("-" * 36)
for ph, counts in sorted_ph:
    print(f"{ph:<10} {counts['total']:<8} {counts['subs']:<6} {counts['dels']:<6} {counts['ins']:<6}")

### 6D — 20 Worst Predictions

In [ ]:
worst = test_results.nlargest(20, 'per')
print("\n--- 20 Worst Predictions (highest PER) ---")
print(f"{'#':<4} {'File':<35} {'Diag':<5} {'PER':<8} {'Canonical':<30} {'Predicted':<30}")
print("-" * 112)
for i, (_, row) in enumerate(worst.iterrows(), 1):
    canonical_str = " ".join(row["canonical"])
    predicted_str = " ".join(row["predicted"]) if row["predicted"] else "(empty)"
    fname = row["filename"][:33]
    print(f"{i:<4} {fname:<35} {row['diagnosis_gold']:<5} {row['per']:.4f}  {canonical_str:<30} {predicted_str:<30}")

### 6E — Inline Plots

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion matrix heatmap
cm = np.array([[tn, fp], [fn, tp]])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred N', 'Pred AB'], yticklabels=['Gold N', 'Gold AB'])
axes[0].set_title('Whole-Word Detection Confusion Matrix')
axes[0].set_ylabel('Gold Label')
axes[0].set_xlabel('Predicted Label')

# 2. Per-phoneme error bar chart (top 15)
if sorted_ph:
    ph_names = [p[0] for p in sorted_ph]
    ph_totals = [p[1]["total"] for p in sorted_ph]
    axes[1].barh(ph_names[::-1], ph_totals[::-1], color='coral')
    axes[1].set_title('Top 15 Phoneme Errors')
    axes[1].set_xlabel('Error Count')

# 3. PER distribution
per_n = test_results.loc[n_mask, 'per'].values
per_ab = test_results.loc[ab_mask, 'per'].values
axes[2].hist(per_n, bins=20, alpha=0.6, label=f'Normal (n={len(per_n)})', color='green')
axes[2].hist(per_ab, bins=20, alpha=0.6, label=f'Abnormal (n={len(per_ab)})', color='red')
axes[2].set_title('PER Distribution: Normal vs Abnormal')
axes[2].set_xlabel('PER')
axes[2].set_ylabel('Count')
axes[2].legend()

plt.tight_layout()
plt.show()

### 6F — Review Table (Top 50 Worst)

In [ ]:
top50 = test_results.nlargest(50, 'per')

html = '<table style="border-collapse:collapse; width:100%; font-size:13px;">'
html += '<tr style="background:#333; color:#fff;"><th>#</th><th>Word</th><th>Diag</th>'
html += '<th>Child</th><th>Canonical</th><th>Predicted</th><th>PER</th><th>Target Op</th></tr>'

for i, (_, row) in enumerate(top50.iterrows(), 1):
    bg = '#ffe0e0' if row['diagnosis_gold'] == 'AB' else '#e0ffe0'
    chip_color = '#c00' if row['target_op'] != 'correct' else '#0a0'
    canonical_str = " ".join(row["canonical"])
    predicted_str = " ".join(row["predicted"]) if row["predicted"] else "(empty)"

    html += f'<tr style="background:{bg};">'
    html += f'<td>{i}</td>'
    html += f'<td>{row["target_word"]}</td>'
    html += f'<td><b>{row["diagnosis_gold"]}</b></td>'
    html += f'<td>{row["id"]}</td>'
    html += f'<td style="font-family:monospace;">{canonical_str}</td>'
    html += f'<td style="font-family:monospace;">{predicted_str}</td>'
    html += f'<td><b>{row["per"]:.3f}</b></td>'
    html += f'<td><span style="background:{chip_color};color:#fff;padding:2px 6px;border-radius:4px;">{row["target_op"]}</span></td>'
    html += '</tr>'

html += '</table>'
display(HTML(html))

---
## Phase 7 — Inference Demo

In [ ]:
def infer(audio_path: str, target_word: str):
    """Run inference on a single audio file."""
    # Load audio
    waveform, sr = torchaudio.load(audio_path)
    if sr != 16000:
        waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if waveform.shape[1] > MAX_SAMPLES:
        waveform = waveform[:, :MAX_SAMPLES]

    # Process
    inputs = processor(waveform.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
    inputs = {k: v.to(trainer.model.device) for k, v in inputs.items()}

    # Forward
    with torch.no_grad():
        logits = trainer.model(**inputs).logits

    # CTC decode
    pred_ids = logits.argmax(-1)[0]
    pred_str = tokenizer.decode(pred_ids, skip_special_tokens=True)
    pred_phonemes = pred_str.split() if pred_str.strip() else []

    # Get canonical
    norm_word = normalize_arabic(target_word)
    if norm_word not in word_to_phonemes:
        print(f"⚠️ '{target_word}' not in lexicon!")
        return
    canonical = word_to_phonemes[norm_word]["phoneme_sequence"]
    target_ph = word_to_phonemes[norm_word]["target_phoneme"]
    target_idx = word_to_phonemes[norm_word]["target_phoneme_idx"]

    # Align
    s, d, ins, ops = levenshtein_align(canonical, pred_phonemes)
    per = (s + d + ins) / len(canonical) if canonical else 0

    # Detection
    has_error = any(o[0] != "correct" for o in ops)
    detection = "Abnormal" if has_error else "Normal"

    # Diagnosis at target
    target_op = diagnose_at_target(pred_phonemes, canonical, target_idx)

    print(f"  Audio:      {os.path.basename(audio_path)}")
    print(f"  Target:     {target_word}")
    print(f"  Canonical:  {' '.join(canonical)}")
    print(f"  Predicted:  {' '.join(pred_phonemes) if pred_phonemes else '(empty)'}")
    print(f"  PER:        {per:.4f}")
    print(f"  Detection:  {detection}")
    print(f"  Diagnosis:  {target_op} (at /{target_ph}/)")
    print()

    return {
        "canonical": canonical, "predicted": pred_phonemes,
        "per": per, "detection": detection, "diagnosis": target_op,
    }

# Demo on one Normal and one Abnormal from test set
test_reset = test_df.reset_index(drop=True)
normal_sample = test_reset[test_reset['diagnosis'] == 'N'].iloc[0]
abnormal_sample = test_reset[test_reset['diagnosis'] == 'AB'].iloc[0]

print("=== Inference Demo: Normal Sample ===")
infer(normal_sample["filepath"], normal_sample["target"])

print("=== Inference Demo: Abnormal Sample ===")
infer(abnormal_sample["filepath"], abnormal_sample["target"])

print("✅ Pipeline complete!")